# Olist Brazilian E-Commerce: Delivery Lead Time Deconstruction & CFO Working Capital Analysis

**Business Context:**  
The Chief Financial Officer (CFO) has issued a mandate to reduce end-to-end order delivery lead times across the Brazilian e-commerce marketplace. Reducing delivery times accelerates inventory turnover, releases working capital locked in supply chain float, and improves customer satisfaction and lifetime value (LTV).

**Analytical Objectives:**
1. **Deconstruct Delivery Lead Times:** Break down total order lead time (`12.56 days` average) into Approval, Warehousing/Fulfillment (`2.85 days`), and Transit (`9.33 days`) phases to identify core bottlenecks.
2. **Quantify Delay Impacts on Repeat Purchases & Reviews:** Analyze how shipping delays damage repeat customer conversion (`3.04%` on-time vs `2.51%` delayed) and review ratings (`4.29` vs `2.56` stars).
3. **CFO Financial Model (`2-Day Transit Reduction`):** Translate a 2-day reduction in average transit time into released balance sheet Working Capital (`$43,253` at dataset scale / `$547,570` at enterprise scale) and recurring Profit & Loss (`EBITDA`) improvements.

## 1. Environment Setup & Data Acquisition

We load the official Olist e-commerce datasets: Orders (`99,441` records), Order Items (`112,650` items), Customers (`99,441` profiles), and Order Reviews (`99,224` reviews). We filter for all completed/delivered orders (`96,478` orders) and join them with transaction values and unique customer identifiers.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Professional plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Load Olist datasets
data_dir = os.path.join("data", "Brazilian E-Commerce Public Dataset by Olist")
orders = pd.read_csv(os.path.join(data_dir, "olist_orders_dataset.csv"))
items = pd.read_csv(os.path.join(data_dir, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(data_dir, "olist_customers_dataset.csv"))
reviews = pd.read_csv(os.path.join(data_dir, "olist_order_reviews_dataset.csv"))

# Filter for delivered orders and aggregate item financial values
df = orders[orders["order_status"] == "delivered"].copy()
order_gmv = items.groupby("order_id").agg(
    price_sum=("price", "sum"),
    freight_sum=("freight_value", "sum")
).reset_index()
order_gmv["gmv"] = order_gmv["price_sum"] + order_gmv["freight_sum"]

df = df.merge(order_gmv, on="order_id", how="left")
df = df.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
order_reviews = reviews.groupby("order_id")["review_score"].mean().reset_index()
df = df.merge(order_reviews, on="order_id", how="left")

# Convert timestamps to datetime
time_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Delivered dataset prepared successfully: {len(df):,} orders across {df['customer_unique_id'].nunique():,} unique customers.")
df.head()

### Analysis of Data Setup
The merged dataset provides end-to-end visibility into all `96,478` delivered transactions between September 2016 and August 2018 (`$15.42M` total Gross Merchandise Value). Each order now maps directly to its financial value (`price_sum` + `freight_sum`), exact phase timestamps, unique human customer ID (`customer_unique_id`), and customer review score.

## 2. Deconstruction of Order Delivery Lead Times

We deconstruct total delivery lead time into three sequential supply chain phases:
1. **Approval Phase:** `order_purchase_timestamp` to `order_approved_at` (payment verification).
2. **Warehousing & Fulfillment Phase:** `order_approved_at` to `order_delivered_carrier_date` (picking, packing, and dispatch).
3. **Transit Phase:** `order_delivered_carrier_date` to `order_delivered_customer_date` (carrier shipping and last-mile delivery).

In [ ]:
df["approval_days"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400.0
df["warehousing_days"] = (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.total_seconds() / 86400.0
df["transit_days"] = (df["order_delivered_customer_date"] - df["order_delivered_carrier_date"]).dt.total_seconds() / 86400.0
df["total_delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400.0
df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.total_seconds() / 86400.0
df["is_delayed"] = df["delay_days"] > 0

phases = [
    ("Approval Phase (Payment Verification)", "approval_days"),
    ("Warehousing Phase (Picking & Packing)", "warehousing_days"),
    ("Transit Phase (Carrier Last-Mile)", "transit_days"),
    ("Total Delivery Lead Time", "total_delivery_days")
]

phase_stats = []
for label, col in phases:
    s = df[col].dropna()
    s = s[s >= 0]  # Exclude minor clock anomalies
    phase_stats.append({
        "Supply Chain Phase": label,
        "Mean (Days)": round(s.mean(), 2),
        "Median (Days)": round(s.median(), 2),
        "Std Dev (Days)": round(s.std(), 2),
        "P90 Tail (Days)": round(s.quantile(0.90), 2),
        "Share of Total Mean (%)": round((s.mean() / df["total_delivery_days"].mean()) * 100, 1) if col != "total_delivery_days" else 100.0
    })

phase_df = pd.DataFrame(phase_stats)
display(phase_df)

### Phase Deconstruction Findings

The empirical breakdown reveals where supply chain capital is locked:
- **Transit Phase is the Dominant Bottleneck:** Carrier shipping and last-mile transit consumes **`9.33 days` (`74.3%` of total delivery time)**. Furthermore, the 90th percentile tail (`P90`) stretches to **`18.90 days`**, indicating severe regional shipping delays.
- **Warehousing/Fulfillment Phase:** Consumes **`2.85 days` (`22.7%` of lead time)**, with a `P90` of `6.02 days`, showing substantial friction in merchant dispatch.
- **Approval Phase:** Minimal friction (`0.43 days` mean, `0.01 days` / 15 minutes median).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart of mean durations
phase_means = [df["approval_days"].clip(lower=0).mean(), df["warehousing_days"].clip(lower=0).mean(), df["transit_days"].clip(lower=0).mean()]
labels = ["Approval\n(0.43d | 3.4%)", "Warehousing / Fulfillment\n(2.80d | 22.3%)", "Transit Phase\n(9.33d | 74.3%)"]
colors = ["#4daf4a", "#377eb8", "#e41a1c"]

ax1.pie(phase_means, labels=labels, colors=colors, autopct="%1.1f%%", startangle=140, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_title("Deconstruction of Total Delivery Lead Time (Mean = 12.56 Days)", fontsize=13, fontweight="bold", pad=15)

# Boxplot comparison of phases
plot_data = pd.DataFrame({
    "Duration (Days)": np.concatenate([
        df["approval_days"].clip(lower=0, upper=30).dropna(),
        df["warehousing_days"].clip(lower=0, upper=30).dropna(),
        df["transit_days"].clip(lower=0, upper=30).dropna()
    ]),
    "Phase": np.concatenate([
        ["1. Approval"] * len(df["approval_days"].clip(lower=0, upper=30).dropna()),
        ["2. Warehousing"] * len(df["warehousing_days"].clip(lower=0, upper=30).dropna()),
        ["3. Transit"] * len(df["transit_days"].clip(lower=0, upper=30).dropna())
    ])
})

sns.boxplot(data=plot_data, x="Phase", y="Duration (Days)", hue="Phase", palette=["#4daf4a", "#377eb8", "#e41a1c"], ax=ax2, width=0.4, fliersize=2, legend=False)
ax2.set_title("Distribution & Tail Latency by Supply Chain Phase (Capped at 30d)", fontsize=13, fontweight="bold", pad=15)
ax2.set_ylabel("Duration (Days)", fontsize=11)
ax2.set_xlabel("Supply Chain Phase", fontsize=11)

plt.show()

## 3. Shipping Delays vs. Customer Repeat Purchase Rate & Review Scores

To quantify how shipping delays impact customer loyalty and brand perception, we track each unique customer (`93,358` individuals) to evaluate whether an initial shipping delay (`order_delivered_customer_date > order_estimated_delivery_date`) degrades subsequent repeat purchase conversion and customer review ratings (`review_score`).

In [ ]:
df_sorted = df.sort_values(["customer_unique_id", "order_purchase_timestamp"]).copy()
customer_agg = df_sorted.groupby("customer_unique_id").agg(
    total_orders=("order_id", "count"),
    first_order_delayed=("is_delayed", "first"),
    first_order_delay_days=("delay_days", "first"),
    first_order_review=("review_score", "first")
).reset_index()

customer_agg["is_repeat_customer"] = customer_agg["total_orders"] > 1
total_unique_customers = len(customer_agg)

status_summary = []
for delayed_flag, group in customer_agg.groupby("first_order_delayed"):
    label = "Delayed Delivery (> 0 days late)" if delayed_flag else "On-Time / Early Delivery (<= 0 days)"
    rep_rate = group["is_repeat_customer"].mean() * 100
    rev_score = group["first_order_review"].mean()
    status_summary.append({
        "Delivery Status": label,
        "Customer Count": f"{len(group):,}",
        "Share of Customers (%)": f"{(len(group)/total_unique_customers)*100:.1f}%",
        "Repeat Purchase Rate (%)": f"{rep_rate:.2f}%",
        "Mean Review Score (1-5)": f"{rev_score:.2f}"
    })

status_df = pd.DataFrame(status_summary)
display(status_df)

# Detailed breakdown across delay severity tiers
bins = [-1000, 0, 3, 7, 1000]
labels = ["1. On Time / Early (<= 0d)", "2. Slight Delay (1-3 days late)", "3. Moderate Delay (4-7 days late)", "4. Severe Delay (> 7 days late)"]
customer_agg["delay_tier"] = pd.cut(customer_agg["first_order_delay_days"], bins=bins, labels=labels)

tier_summary = []
for tier, group in customer_agg.groupby("delay_tier", observed=False):
    rep_rate = group["is_repeat_customer"].mean() * 100
    rev_score = group["first_order_review"].mean()
    tier_summary.append({
        "Delay Severity Tier": str(tier),
        "Customer Count": f"{len(group):,}",
        "Repeat Purchase Rate (%)": round(rep_rate, 2),
        "Mean Review Score": round(rev_score, 2)
    })
tier_df = pd.DataFrame(tier_summary)
display(tier_df)

### Customer Conversion & Review Analysis

The data proves that shipping delays inflict significant commercial damage:
1. **Repeat Purchase Penalty (`17.4% Relative Drop`):** Customers who receive their order on-time return at a **`3.04%` repeat rate**. When delivery is delayed (`> 0 days late`), repeat conversion drops to **`2.51%`**—a relative churn penalty of **`17.4%`** (`(3.04 - 2.51) / 3.04`).
2. **Review Score Degradation (`40.3% Drop`):** On-time customers award high ratings (**`4.29 out of 5 stars`**). When delayed, the average score collapses to **`2.56 stars`**, and for severe delays (`> 7 days`), it plummets to **`1.72 stars`**. Lower review scores impair organic product ranking and future customer acquisition.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot of repeat rates
sns.barplot(data=tier_df, x="Delay Severity Tier", y="Repeat Purchase Rate (%)", hue="Delay Severity Tier", palette="Blues_r", ax=ax1, legend=False)
ax1.set_title("Customer Repeat Purchase Rate by Initial Delivery Experience", fontsize=13, fontweight="bold", pad=15)
ax1.set_ylabel("Repeat Purchase Rate (%)", fontsize=11)
ax1.set_xlabel("Delivery Delay Tier", fontsize=11)
ax1.tick_params(axis='x', rotation=15)
for p in ax1.patches:
    height = p.get_height()
    ax1.annotate(f"{height:.2f}%", (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3), textcoords='offset points')

# Bar plot of review scores
sns.barplot(data=tier_df, x="Delay Severity Tier", y="Mean Review Score", hue="Delay Severity Tier", palette="Reds_r", ax=ax2, legend=False)
ax2.set_title("Average Customer Review Score by Initial Delivery Experience", fontsize=13, fontweight="bold", pad=15)
ax2.set_ylabel("Review Score (Out of 5 Stars)", fontsize=11)
ax2.set_xlabel("Delivery Delay Tier", fontsize=11)
ax2.set_ylim(0, 5)
ax2.tick_params(axis='x', rotation=15)
for p in ax2.patches:
    height = p.get_height()
    ax2.annotate(f"{height:.2f}", (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3), textcoords='offset points')

plt.show()

## 4. CFO Financial Model: 2-Day Transit Reduction Working Capital & EBITDA Impact

We model the exact financial benefits of achieving a **2-day reduction in average transit time** across both Balance Sheet Working Capital and ongoing Profit & Loss (`EBITDA`).

### Financial Mechanisms & Assumptions:
1. **Working Capital Float Released:** Goods in transit tie up working capital (inventory float / escrow payment delay). Releasing **2 days of transit time** permanently frees up **2 days of Daily GMV Run-Rate** from working capital float onto the balance sheet.
2. **Carrying Cost / Financing Savings (`EBITDA Lever 1`):** At a Weighted Average Cost of Capital (`WACC`) / inventory holding cost rate of **`12.0% per annum`**, freeing working capital generates immediate annual financing/carrying cost reductions.
3. **Customer Retention EBITDA Uplift (`EBITDA Lever 2`):** Shifting the delivery distribution faster by `2 days` turns **`2,117 currently delayed orders` (`27.05% of all delays`) into on-time deliveries**, lifting their repeat conversion rate (`+0.53%`) and review scores (`4.29` vs `2.56`). Assuming an average order item value of **`$137.04`** and a **`20% EBITDA contribution margin`**, we quantify the recurring profit gain.

In [ ]:
min_date = df["order_purchase_timestamp"].min()
max_date = df["order_purchase_timestamp"].max()
total_days = (max_date - min_date).days
total_gmv = df["gmv"].sum()
daily_gmv = total_gmv / total_days
annual_gmv = daily_gmv * 365.25

# 1. Working Capital Released (2 days of float)
wc_released_dataset = daily_gmv * 2.0
enterprise_annual_gmv = 100_000_000.0
wc_released_enterprise = (enterprise_annual_gmv / 365.25) * 2.0

# 2. EBITDA Carrying Cost Savings (12% WACC)
wacc_rate = 0.12
ebitda_carrying_savings_dataset = wc_released_dataset * wacc_rate
ebitda_carrying_savings_enterprise = wc_released_enterprise * wacc_rate

# 3. Repeat Customer EBITDA Uplift (curing 27.05% of delays)
df["delay_days_after_2d_cut"] = df["delay_days"] - 2.0
df["is_delayed_after"] = df["delay_days_after_2d_cut"] > 0
delayed_before_count = df["is_delayed"].sum()
delayed_after_count = df["is_delayed_after"].sum()
cured_orders_count = delayed_before_count - delayed_after_count

repeat_rate_uplift = (0.0304 - 0.0251)
avg_order_price = df["price_sum"].mean()
ebitda_margin_on_gmv = 0.20

cured_orders_annual = cured_orders_count * (365.25 / total_days)
incremental_repeat_orders_annual = cured_orders_annual * repeat_rate_uplift
incremental_repeat_gmv_annual = incremental_repeat_orders_annual * avg_order_price
ebitda_repeat_uplift_annual = incremental_repeat_gmv_annual * ebitda_margin_on_gmv
total_annual_ebitda_impact = ebitda_carrying_savings_dataset + ebitda_repeat_uplift_annual

print("CFO WORKING CAPITAL & EBITDA IMPACT SUMMARY:")
print("-" * 80)
print(f"A. WORKING CAPITAL RELEASED (One-Time Balance Sheet Cash Flow Improvement):")
print(f"   - Olist Dataset Scale (${annual_gmv:,.0f}/yr GMV) : ${wc_released_dataset:,.2f} freed working capital")
print(f"   - Enterprise Benchmark (${enterprise_annual_gmv:,.0f}/yr GMV): ${wc_released_enterprise:,.2f} freed working capital\n")

print(f"B. ONGOING ANNUAL EBITDA IMPACT (Profit & Loss Statement Uplift):")
print(f"   1. Carrying Cost / Financing Savings (12% WACC on Freed Capital):")
print(f"      - Olist Dataset Scale : +${ebitda_carrying_savings_dataset:,.2f} / year")
print(f"      - Enterprise Scale    : +${ebitda_carrying_savings_enterprise:,.2f} / year\n")
print(f"   2. Customer Retention EBITDA Uplift (From Curing {(cured_orders_count/delayed_before_count)*100:.1f}% of Delays):")
print(f"      - Delayed Orders Cured to On-Time : {cured_orders_count:,} orders ({cured_orders_annual:,.0f}/year annualized)")
print(f"      - Repeat Purchase Rate Increase   : +0.53% (from 2.51% to 3.04%)")
print(f"      - Incremental Repeat GMV          : +${incremental_repeat_gmv_annual:,.2f} / year")
print(f"      - Incremental Repeat EBITDA (20%) : +${ebitda_repeat_uplift_annual:,.2f} / year\n")
print(f"   TOTAL ANNUAL EBITDA IMPACT (Dataset Scale) : +${total_annual_ebitda_impact:,.2f} / year")
print("=" * 80)

### CFO Economic Summary

Reducing average transit time by 2 days delivers powerful balance sheet and income statement benefits:
1. **Balance Sheet Cash Flow (Freed Working Capital Float):**  
   Releasing 2 days of transit time immediately unlocks **`$43,253.22` in cash flow from working capital float** (`$547,570` at `$100M/year GMV scale`).
2. **Ongoing Profit & Loss (`EBITDA`) Improvement:**  
   - **Carrying Cost Savings:** Eliminating the financing carrying cost of freed float contributes **`+$5,190.39/year`** in recurring EBITDA (`+$65,708/year` at `$100M GMV scale`).
   - **Repeat Conversion Revenue & EBITDA:** The 2-day transit cut **cures `2,117` currently delayed orders (`27.05%` of all shipping delays)** into on-time deliveries, increasing customer repeat conversion and contributing **`+$157.54/year`** in direct repeat order EBITDA margin.
   - **Total Annual EBITDA Impact:** **`+$5,347.92/year`** (`+$67,700+/year` at enterprise benchmark scale).

## 5. Comprehensive Summary & CFO Recommendations

### Key Analytical Findings
- **Transit Phase Bottleneck:** Carrier transit represents **`74.3%` (`9.33 days`)** of total delivery lead time (`12.56 days`), whereas warehousing/fulfillment accounts for `22.7%` (`2.85 days`) and approval accounts for `3.4%` (`0.43 days`).
- **Delay Damage on Repeat Conversion & Reviews:** Shipping delays inflict a **`17.4%` relative drop in repeat customer conversion** (`3.04%` on time vs `2.51%` delayed) and a **`40.3%` collapse in average review ratings** (`4.29` vs `2.56` stars).
- **Working Capital & EBITDA Payoff (`2-Day Reduction`):** Shifting transit times faster by 2 days unlocks **`$43,253.22` in balance sheet working capital** and generates **`+$5,347.92/year` in recurring EBITDA contribution** by saving carrying costs and curing `27.05%` of delayed deliveries.

### Strategic Action Plan for the CFO & Operations Leadership
1. **Negotiate Regional Carrier SLAs (`Targeting the 74.3% Bottleneck`):** Focus CapEx and contractual incentives on regional carriers rather than payment verification. Structure carrier bonuses around reducing last-mile transit times below `7 days`.
2. **Implement Regional Fulfillment Hubs (`Cross-Docking`):** Establish regional fulfillment nodes in high-density customer states (`SP`, `RJ`, `MG`) to bypass long-haul interstate transit entirely, curing the `P90` delay tail (`18.90 days`).
3. **Deploy Real-Time Delay Interventions:** Monitor orders approaching their estimated delivery date. Automatically issue proactive status communications or expedited carrier upgrades 2 days before SLA breach to preserve customer review scores and repeat conversion.